# 2020–21 CRDC data cleaning

Step-by-step walkthrough of `scripts/clean_202021.py`. It produces the shared 50-column school table used by the other years (`AI_CONTEXT.md` section A4, `cleaning_feature_playbook.md`), so the years can be stacked for modeling.

2020–21 is the COVID year. Seclusion reporting is incomplete: many schools are DQ-suppressed (`-11`) and virtual-only schools are skipped (`-13`). Use it for robustness / sensitivity analysis, not as the primary modeling year. Negative reserved codes are missingness, not counts: they are set to NA and never filled with 0. The original codes go to the problem log (`quality_detail_202021.csv`), not the main file.

Outputs: `schools_202021.csv`, `schools_202021_analysis.csv`, `quality_detail_202021.csv`, `covid_indicators_202021.csv` (COVID columns, keyed by `combokey`), plus `docs/cleaning/qa_202021.md` and `mapping_202021.csv`.

Restraint columns (`MECH` / `PHYS`) are deliberately not read: the EDA found restraint leaks the outcome.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))

import clean_202021 as cc

RAW_DIR = cc.DEFAULT_RAW_DIR      # data/raw/2020-21
OUT_DIR = cc.DEFAULT_OUT_DIR      # data/processed/202021 (not tracked by git)
DOCS_DIR = cc.DEFAULT_DOCS_DIR    # docs/cleaning

cc.run_self_checks()

## 1–3. Read, check IDs, join

- Files are read as strings (UTF-8, falling back to `cp1252`) so IDs keep leading zeros.
- IDs are checked for blank, non-digit and too-long values before padding `LEAID` to 7, `SCHID` to 5 and `COMBOKEY` to 12. Non-digit IDs are kept as reported and logged.
- School Characteristics is the roster. Enrollment, Restraint and Seclusion, School Support and COVID Directional Indicators are left-joined on `combokey` (`one_to_one`), and LEA Characteristics on `leaid` (`many_to_one`). Missing and extra keys are logged. Duplicate keys stop the run.
- `lea_name` comes from the LEA file; disagreements with the school file are logged.

In [ ]:
df, ctx = cc.load_and_join(RAW_DIR)
pd.DataFrame(ctx["joins"])

## 4–5. Recode reserved codes and build the shared columns

- Values `< 0` become NA and each one is written to the problem log with its raw code and reason group. Zeros stay zeros.
- `enrollment_total` = `TOT_ENR_M` + `TOT_ENR_F` (NA if either is NA). `enr_nonbinary` is always NA. `enr_el` comes from the LEP columns.
- `seclusion_instances` = WODIS + IDEA + 504, only when all three are valid. `seclusion_rate` is per 100 students, NA when enrollment is missing or 0.
- `pct_idea` / `pct_504` / `pct_el` are on a 0–100 scale and NA when the subgroup is larger than the total.
- `support_fte_per_100` = nurses + psychologists + social workers per 100 students (counselors not included).
- FTE is flagged for review when teachers > 2,000 or any other core FTE > 200. Values are never changed.
- `flag_small_enrollment` is under 10 students; `flag_analysis_ready` needs a valid seclusion total and enrollment above 0.
- `sch_virt_ind` is set from the COVID instruction type as requested in the PR review (the playbook says to leave it NA). Change `SCH_VIRT_FROM_COVID` in the script to switch.

In [ ]:
result = cc.recode_and_build(df, ctx)
out = result["out"]
print(out.shape)
out.head()

## Audit checks

In [ ]:
report = cc.audit_report(result)
cc.print_audit_checks(report)
assert list(out.columns) == cc.OUTPUT_COLUMNS and len(out.columns) == 50

## COVID-year seclusion gaps

In [ ]:
work = result["work"]
pd.crosstab(
    work["SCH_DIND_INSTRUCTIONTYPE"].fillna("NA"),
    out["flag_missing_seclusion"].map({True: "missing seclusion", False: "valid seclusion"}),
)

## Write outputs, check them, and write the docs

In [ ]:
paths = cc.write_outputs(result, OUT_DIR)
cc.validate_outputs(result, paths)
cc.write_mapping(DOCS_DIR / "mapping_202021.csv")
cc.write_qa(result, report, DOCS_DIR / "qa_202021.md", RAW_DIR)